In [1]:
# imports

import os
from dotenv import load_dotenv
from huggingface_hub import login
from pricer.evaluator import evaluate
from litellm import completion
from pricer.items import Item
import numpy as np
from tqdm.notebook import tqdm
import csv
from sklearn.feature_extraction.text import HashingVectorizer
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import CosineAnnealingLR


In [7]:
import sys
!{sys.executable} -m pip install torch

  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/114.5 MB ? eta -:--:--
   ---------------------------------------- 0.2/114.5 MB 3.1 MB/s eta 0:00:38
   ---------------------------------------- 0.3/114.5 MB 3.5 MB/s eta 0:00:33
   ---------------------------------------- 0.5/114.5 MB 3.5 MB/s eta 0:00:33
   ---------------------------------------- 0.7/114.5 MB 3.6 MB/s eta 0:00:32
   ---------------------------------------- 0.9/114.5 MB 3.8 MB/s eta 0:00:30
   ---------------------------------------- 1.1/114.5 MB 3.7 MB/s eta 0:00:31
   ---------------------------------------- 1.2/114.5 MB 3.9 MB/s eta 0:00:29
    --------------------------------------- 1.4/114.5 MB 3.8 MB/s eta 0:00:30
    --------------------------------------- 1.6/114.5 MB 3.8 MB/s eta 0:00:30
    ---------------------------------


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
LITE_MODE = True

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Loaded 20,000 training items, 1,000 validation items, 1,000 test items


In [ ]:
# Write the test set to a CSV

with open('human_in.csv', 'w', encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    for t in test[:100]:
        writer.writerow([t.summary, 0])
        

And now vanilla newlork network


In [9]:
# Prepare our documents and prices

y = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

In [10]:
# Use the HashingVectorizer for a Bag of Words model
# Using binary=True with the CountVectorizer makes "one-hot vectors"

np.random.seed(42)
vectorizer = HashingVectorizer(n_features=5000, stop_words='english', binary=True)
X = vectorizer.fit_transform(documents)

In [11]:
# Define the neural network - here is Pytorch code to create a 8 layer neural network

class NeuralNetwork(nn.Module):
    def __init__(self, input_size):
        super(NeuralNetwork, self).__init__()
        self.layer1 = nn.Linear(input_size, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 64)
        self.layer4 = nn.Linear(64, 64)
        self.layer5 = nn.Linear(64, 64)
        self.layer6 = nn.Linear(64, 64)
        self.layer7 = nn.Linear(64, 64)
        self.layer8 = nn.Linear(64, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        output1 = self.relu(self.layer1(x))
        output2 = self.relu(self.layer2(output1))
        output3 = self.relu(self.layer3(output2))
        output4 = self.relu(self.layer4(output3))
        output5 = self.relu(self.layer5(output4))
        output6 = self.relu(self.layer6(output5))
        output7 = self.relu(self.layer7(output6))
        output8 = self.layer8(output7)
        return output8

In [12]:
# Convert data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X.toarray())
y_train_tensor = torch.FloatTensor(y).unsqueeze(1)

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train_tensor, y_train_tensor, test_size=0.01, random_state=42)

# Create the loader
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Initialize the model
input_size = X_train_tensor.shape[1]
model = NeuralNetwork(input_size)

In [13]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Number of trainable parameters: {trainable_params:,}")

Number of trainable parameters: 669,249


In [14]:
# Define loss function and optimizer

loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# We will do 2 complete runs through the data

EPOCHS = 2

for epoch in range(EPOCHS):
    model.train()
    for batch_X, batch_y in tqdm(train_loader):
        optimizer.zero_grad()

        # The next 4 lines are the 4 stages of training: forward pass, loss calculation, backward pass, optimize
        
        outputs = model(batch_X)
        loss = loss_function(outputs, batch_y)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val)
        val_loss = loss_function(val_outputs, y_val)

    print(f'Epoch [{epoch+1}/{EPOCHS}], Train Loss: {loss.item():.3f}, Val Loss: {val_loss.item():.3f}')

  0%|          | 0/310 [00:00<?, ?it/s]

Epoch [1/2], Train Loss: 10644.841, Val Loss: 20210.666


  0%|          | 0/310 [00:00<?, ?it/s]

Epoch [2/2], Train Loss: 12575.114, Val Loss: 18828.135


In [15]:
def neural_network(item):
    model.eval()
    with torch.no_grad():
        vector = vectorizer.transform([item.summary])
        vector = torch.FloatTensor(vector.toarray())
        result = model(vector)[0].item()
    return max(0, result)

In [16]:
evaluate(neural_network, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$80 $52 $17 $23 $40 $175 $34 $57 $34 $15 $490 $156 $108 $145 $25 $32 $1 $28 $99 $24 $35 $26 $68 $46 $274 $279 $260 $34 $19 $39 $50 $128 $39 $25 $82 $284 $50 $112 $107 $58 $160 $50 $26 $97 $126 $48 $77 $55 $7 $35 $12 $40 $146 $16 $103 $43 $46 $128 $6 $34 $90 $7 $37 $8 $475 $90 $45 $288 $6 $213 $5 $33 $137 $97 $14 $51 $44 $44 $33 $65 $63 $130 $28 $30 $14 $70 $57 $167 $89 $108 $21 $48 $28 $20 $37 $104 $62 $12 $138 $253 $27 $64 $0 $35 $34 $49 $93 $264 $5 $93 $6 $99 $120 $6 $16 $81 $124 $41 $55 $36 $21 $190 $27 $0 $75 $25 $24 $178 $16 $49 $46 $108 $102 $33 $71 $20 $96 $75 $44 $58 $29 $153 $11 $158 $156 $70 $66 $299 $99 $17 $18 $196 $14 $70 $37 $138 $129 $8 $40 $7 $119 $9 $0 $24 $542 $18 $96 $9 $19 $48 $12 $20 $267 $49 $13 $2 $20 $1 $27 $76 $414 $18 $118 $28 $51 $85 $31 $56 $31 $11 $63 $64 $76 $9 $6 $29 $42 $53 $1 $16 

now frontier model

In [17]:
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": message}]

In [18]:
print(test[0].summary)

Title: Excess V2 Distortion/Modulation Pedal  
Category: Music Pedals  
Brand: Old Blood Noise  
Description: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  
Details: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.


In [19]:
messages_for(test[0])

[{'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Excess V2 Distortion/Modulation Pedal  \nCategory: Music Pedals  \nBrand: Old Blood Noise  \nDescription: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  \nDetails: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.'}]

In [20]:
# The function for gpt-4.1-nano

def gpt_4__1_nano(item):
    response = completion(model="openai/gpt-4.1-nano", messages=messages_for(item))
    return response.choices[0].message.content

In [21]:
gpt_4__1_nano(test[0])

'$200'

In [22]:
test[0].price

219.0

In [23]:
evaluate(gpt_4__1_nano, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$31 $84 $30 $0 $45 $80 $6 $65 $10 $870 $213 $20 $30 $4 $19 $8 $71 $5 $40 $1 $64 $26 $65 $175 $182 $264 $305 $5 $351 $65 $30 $15 $10 $50 $35 $219 $60 $21 $14 $13 $175 $35 $25 $105 $70 $5 $17 $13 $75 $52 $20 $105 $125 $9 $147 $16 $8 $40 $2 $3 $116 $33 $31 $40 $21 $30 $90 $295 $25 $74 $16 $8 $30 $6 $15 $21 $126 $0 $8 $3 $30 $3 $5 $74 $11 $0 $32 $44 $0 $4 $13 $55 $5 $20 $2 $78 $1 $7 $70 $375 $20 $63 $11 $11 $51 $32 $10 $380 $19 $49 $20 $36 $29 $53 $34 $130 $0 $7 $64 $47 $24 $261 $60 $16 $50 $10 $5 $151 $29 $64 $79 $13 $65 $10 $85 $0 $55 $10 $78 $62 $86 $150 $70 $12 $134 $88 $15 $290 $115 $13 $6 $194 $17 $10 $4 $129 $101 $41 $70 $5 $311 $17 $7 $3 $340 $7 $752 $30 $15 $5 $0 $3 $120 $8 $22 $201 $3 $57 $4 $23 $546 $15 $150 $1 $10 $13 $73 $17 $10 $12 $5 $99 $5 $11 $20 $40 $30 $30 $1 $1 